### 🧠 Sliding Window Causal Attention — Detailed Explanation

Sliding Window Attention is a modification of **standard transformer attention** that limits how far each token can attend.

Instead of attending to **all previous tokens**, each token attends to **only a fixed window of recent tokens**.

This reduces computation from:

$$
O(T^2)
$$

to

$$
O(T \times W)
$$

where:

* **T** = sequence length
* **W** = sliding window size

---

### ⚙️ Reference Code

```python
class SlidingWindowCausalAttention(nn.Module):

    def __init__(self, embed_dim, num_heads, window_size, dropout=0.0, bias=False):
```

---

### 🏗 Step 1 — Initialize the Attention Module

The class defines a **PyTorch module** implementing sliding window causal attention.

It will be used inside a **Transformer block**.

Parameters:

| Parameter   | Meaning                   |
| ----------- | ------------------------- |
| embed_dim   | embedding dimension       |
| num_heads   | number of attention heads |
| window_size | maximum tokens visible    |
| dropout     | dropout probability       |

Example configuration:

```
embed_dim = 4096
num_heads = 32
window_size = 4096
```

Used in models like **Mistral / Gemma**.

---

### 📐 Step 2 — Head Dimension Calculation

```python
self.head_dim = embed_dim // num_heads
```

Each attention head processes a **smaller subspace** of the embedding.

Formula:

$$
head_dim = \frac{embed_dim}{num_heads}
$$

Example:

```
embed_dim = 4096
num_heads = 32
```

Result:

```
head_dim = 128
```

---

### ⚖️ Step 3 — Attention Scaling Factor

```python
self.scale = self.head_dim ** -0.5
```

Attention scores can become very large if not scaled.

The scaling factor stabilizes training.

Formula:

$$
scale = \frac{1}{\sqrt{d_k}}
$$

Where:

* $d_k$ = head dimension

Example:

```
head_dim = 128
scale = 1 / sqrt(128)
```

---

### 🔧 Step 4 — Query, Key, Value Projections

```python
self.q_proj = nn.Linear(embed_dim, embed_dim)
self.k_proj = nn.Linear(embed_dim, embed_dim)
self.v_proj = nn.Linear(embed_dim, embed_dim)
```

These layers generate **Query, Key, and Value vectors**.

Formulas:

$$
Q = XW_Q
$$

$$
K = XW_K
$$

$$
V = XW_V
$$

Input shape:

```
(B, T, C)
```

Where:

* **B** = batch size
* **T** = sequence length
* **C** = embedding dimension

---

### 🔄 Step 5 — Output Projection

```python
self.out_proj = nn.Linear(embed_dim, embed_dim)
```

After computing attention, we project back to embedding space.

Formula:

$$
Output = Attention \times W_O
$$

---

### 📦 Step 6 — Forward Pass

```python
def forward(self, x):
```

Input tensor shape:

```
(B, T, C)
```

Example:

```
B = 2
T = 1024
C = 512
```

---

### 📊 Step 7 — Extract Tensor Dimensions

```python
B, T, C = x.shape
```

This gives:

| Symbol | Meaning             |
| ------ | ------------------- |
| B      | batch size          |
| T      | sequence length     |
| C      | embedding dimension |

---

### 🔎 Step 8 — Compute Q, K, V

```python
q = self.q_proj(x)
k = self.k_proj(x)
v = self.v_proj(x)
```

Shapes remain:

```
(B, T, C)
```

Each token now has:

* Query vector
* Key vector
* Value vector

---

### 🧩 Step 9 — Split Into Multiple Heads

```python
q = q.view(B, T, num_heads, head_dim).transpose(1, 2)
```

Before reshape:

```
(B, T, C)
```

After reshape:

```
(B, T, H, D)
```

After transpose:

```
(B, H, T, D)
```

Where:

* **H** = number of heads
* **D** = head dimension

This is done for **Q, K, and V**.

---

### 🧮 Step 10 — Compute Raw Attention Scores

```python
attn = torch.matmul(q, k.transpose(-2, -1)) * self.scale
```

Mathematical formula:

$$
Attention = \frac{QK^T}{\sqrt{d_k}}
$$

Tensor shapes:

```
q : (B, H, T, D)
k : (B, H, T, D)
```

Result:

```
(B, H, T, T)
```

This matrix represents **how much each token attends to every other token**.

---

### 🪟 Step 11 — Build Sliding Window Mask

```python
mask = self.build_sliding_mask(T, x.device)
```

The mask ensures:

1️⃣ **No future tokens (causal)**
2️⃣ **Only last W tokens visible**

---

### 🔍 Step 12 — Create Token Index Vector

```python
idx = torch.arange(T)
```

Example:

```
T = 8
```

Result:

```
[0 1 2 3 4 5 6 7]
```

Each number represents a **token position**.

---

### ⚖️ Step 13 — Build Causal Mask

```python
causal = idx.unsqueeze(1) >= idx.unsqueeze(0)
```

This creates a **lower triangular matrix**.

Example:

```
T0 [1 0 0 0]
T1 [1 1 0 0]
T2 [1 1 1 0]
T3 [1 1 1 1]
```

Meaning:

```
future tokens are hidden
```

---

### 🪟 Step 14 — Build Window Constraint

```python
window = (idx.unsqueeze(1) - idx.unsqueeze(0)) < window_size
```

This restricts attention to **recent tokens only**.

Formula:

$$
i - j < W
$$

Where:

* **i** = query token
* **j** = key token

---

### 🔗 Step 15 — Combine Masks

```python
mask = causal & window
```

This ensures both conditions:

✔ no future tokens
✔ limited context window

---

### 🧠 Step 16 — Example Sliding Window Mask

Example with:

```
window = 4
T = 8
```

Matrix:

```
      T0 T1 T2 T3 T4 T5 T6 T7
T0    1
T1    1 1
T2    1 1 1
T3    1 1 1 1
T4      1 1 1 1
T5        1 1 1 1
T6          1 1 1 1
T7            1 1 1 1
```

This matches exactly the sliding window pattern you wanted.

---

### 🚫 Step 17 — Apply Mask to Attention Scores

```python
attn = attn.masked_fill(~mask, -inf)
```

Masked values become:

```
-softmax → 0
```

Meaning the model **cannot attend to those tokens**.

---

### 🎯 Step 18 — Softmax Normalization

```python
attn = F.softmax(attn, dim=-1)
```

This converts scores into probabilities.

Formula:

$$
softmax(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}
$$

Each row now sums to **1**.

---

### 💧 Step 19 — Apply Dropout

```python
attn = self.dropout(attn)
```

Dropout prevents **overfitting** during training.

---

### 🧮 Step 20 — Weighted Value Aggregation

```python
out = torch.matmul(attn, v)
```

Formula:

$$
Output = Attention \times V
$$

Shape:

```
(B, H, T, D)
```

Each token now contains **context information from nearby tokens**.

---

### 🔄 Step 21 — Merge Attention Heads

```python
out = out.transpose(1, 2).contiguous().view(B, T, C)
```

Before merge:

```
(B, H, T, D)
```

After merge:

```
(B, T, C)
```

Heads are **concatenated back together**.

---

### 🧱 Step 22 — Final Linear Projection

```python
out = self.out_proj(out)
```

Final transformation:

$$
Output = Attention \times W_O
$$

Output shape:

```
(B, T, C)
```

---

# 📊 Complexity Comparison

### Full Attention

$$
O(T^2)
$$

Example:

```
T = 8192
```

Matrix size:

```
8192 × 8192 = 67M
```

---

### Sliding Window Attention

$$
O(T \times W)
$$

Example:

```
T = 8192
W = 512
```

Matrix:

```
8192 × 512 = 4.1M
```

Memory reduction:

```
≈ 16× smaller
```

---

# 🚀 Why Sliding Window Attention Is Important

Advantages:

✅ Handles **very long sequences**
✅ Reduces **memory consumption**
✅ Faster training for long contexts

Used in models such as:

* **Longformer**
* **Mistral**
* **Gemma**
* **Mixtral**

---

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class SlidingWindowCausalAttention(nn.Module):

    def __init__(self, embed_dim, num_heads, window_size, dropout=0.0, bias=False):
        super().__init__()

        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.head_dim = embed_dim // num_heads

        self.scale = self.head_dim ** -0.5

        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=bias)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=bias)
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=bias)

        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=bias)

        self.dropout = nn.Dropout(dropout)

    def build_sliding_mask(self, T, device):

        idx = torch.arange(T, device=device)

        causal = idx.unsqueeze(1) >= idx.unsqueeze(0)

        window = (idx.unsqueeze(1) - idx.unsqueeze(0)) < self.window_size

        mask = causal & window

        return mask

    def forward(self, x):

        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # attention scores
        attn = torch.matmul(q, k.transpose(-2, -1)) * self.scale

        # sliding window mask
        mask = self.build_sliding_mask(T, x.device)

        attn = attn.masked_fill(~mask, float("-inf"))

        attn = F.softmax(attn, dim=-1)

        attn = self.dropout(attn)

        out = torch.matmul(attn, v)

        out = out.transpose(1, 2).contiguous().view(B, T, C)

        out = self.out_proj(out)

        return out